# Feature Engineering — IEEE-CIS Fraud Detection

**Objetivo de este notebook:** crear nuevas variables a partir de los hallazgos del análisis exploratorio, para que los modelos puedan detectar mejor el fraude.

En el notebook 01 comprobamos que ninguna variable explica el fraude por sí sola (la correlación máxima con el target es de 0,15), pero que hay señales claras en la ausencia de datos, en el importe, en algunas categorías y en las interacciones entre variables. En este notebook convertimos esas señales en variables:
- **Separación temporal** entre entrenamiento y validación, antes de crear cualquier variable que aprenda de los datos.
- **Variables por reglas fijas:** indicadores de ausencia de datos, familias de sistema operativo y navegador, marca del dispositivo, variables de tiempo y transformaciones del importe.
- **Variables agregadas:** resúmenes del comportamiento de cada tarjeta, calculados sin usar información del periodo de validación.
- **Comprobación:** relación de cada variable nueva con el fraude.

Guardamos los conjuntos de entrenamiento, validación y test con las nuevas variables. El tratamiento de nulos, el encoding y el escalado dependen de cada modelo, por lo que se aplicarán dentro de los pipelines del notebook de modelado.

In [14]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [15]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid", palette="deep", rc={"figure.figsize": (10, 5)})

sorted(p.name for p in PROCESSED_DIR .glob("*.parquet"))

['test_clean.parquet', 'train_clean.parquet']

In [16]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")

print(train.shape)
print(test.shape)

(590537, 434)
(506691, 433)


## 1. Split

In [17]:
corte = train["TransactionDT"].quantile(0.8)
train["split"] = np.where(train["TransactionDT"] <= corte, "train", "valid")
test["split"] = "test"

df = pd.concat([train, test], ignore_index=True)
del train, test

df["split"].value_counts()

split
test     506691
train    472429
valid    118108
Name: count, dtype: int64

Split temporal, hay que ordenar por fecha, unimos todo para asi no hacer cada accion tres veces

## 2.Nulos

Descubrimos que cada modelo decidirá que hacer con los nulos, en este caso haremos solo 3 que vimos que si influian

In [20]:
df['has_identity'] = df['id_12'].notna()
df['has_addr']= df['addr1'].notna()
df['has_r_email'] = df['R_emaildomain'].notna()



**Sistema operativo**

In [19]:
df["os_family"] = df["id_30"].str.split().str[0]
df["os_family"] = df["id_30"].str.split().str[0]

In [23]:
print(df["os_family"].unique())
df[df["split"] == "train"].groupby("os_family", dropna=False)["isFraud"].mean()

[nan 'android' 'ios' 'mac' 'windows' 'linux' 'func' 'other']


os_family
android   0.086
func      0.000
ios       0.061
linux     0.082
mac       0.018
other     0.333
windows   0.035
NaN       0.034
Name: isFraud, dtype: float64

## Familia del sistema operativo

`id_30` contiene la versión exacta del sistema operativo (`windows 10`, `ios 11.4.1`, `mac os x 10_14_6`…). Hay decenas de versiones y muchas tienen muy pocas transacciones, así que el modelo no puede aprender un patrón fiable de cada una. Agrupamos las versiones en familias quedándonos con la primera palabra, de modo que cada grupo reúne suficientes filas.

Calculamos la tasa de fraude por familia **solo con las filas de entrenamiento**, para no usar información de validación ni de test:

- **Android (8,6%), Linux (8,2%) e iOS (6,1%)** superan con claridad la tasa general del 3,5%.
- **Windows (3,5%)** está en la media y **Mac (1,8%)** queda por debajo.
- **`other` (33,3%) y `func` (0%)** muestran valores extremos, pero se calculan sobre muy pocas filas (XX y XX), por lo que no son fiables. `func` es un valor sin documentar en el dataset original.

**Conclusión:** la familia del sistema

**Navegador**

In [25]:
df["browser_family"] = df["id_31"].str.split().str[0]
df["browser_family"] = df["browser_family"].replace({"mobile": "safari", "mozilla/firefox": "firefox"})

counts = df[df["split"] == "train"]["browser_family"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["browser_family"].isin(keep) & df["browser_family"].notna(), "browser_family"] = "other"

df[df["split"] == "train"].groupby("browser_family", dropna=False)["isFraud"].agg(["mean", "count"])
  

,mean,count
browser_family,,
chrome,0.093,63376
edge,0.034,4614
firefox,0.071,6059
ie,0.018,8454
other,0.228,1300
safari,0.062,31994
samsung,0.083,1666
NaN,0.021,354966


## Familia del navegador

`id_31` contiene el navegador y su versión (`chrome 63.0`, `mobile safari 11.0`…). Igual que con el sistema operativo, nos quedamos con la primera palabra para agrupar las versiones en familias, y corregimos los dos casos en los que esa regla falla: `mobile safari` pasa a `safari` y `mozilla/firefox` a `firefox`.

Muchas familias tienen muy pocas transacciones y tasas de fraude extremas (0% o 100%) que son ruido, no señal. Para evitar que el modelo aprenda esas casualidades (overfitting), **conservamos solo las familias con al menos 500 transacciones en entrenamiento**, que con una tasa base del 3,5% equivalen a unos 17 fraudes esperados. El resto se agrupa en `other`. Al definir la lista de familias que conservamos, y no la de las raras, los navegadores que aparezcan por primera vez en test también se asignan a `other`.

Resultados en entrenamiento:

- **`other` (22,8%)** tiene la tasa más alta. Incluye navegadores poco comunes de móviles Android, que por separado ya mostraban tasas muy elevadas.
- **Chrome (9,3%), Samsung (8,3%), Firefox (7,1%) y Safari (6,2%)** están por encima de la media del 3,5%. Destaca Chrome, por combinar una tasa casi tres veces superior a la media con el mayor volumen de transacciones.
- **Edge (3,4%) e Internet Explorer (1,8%)** están en la media o por debajo.
- Las transacciones **sin navegador (2,1%)** coinciden con las que no tienen datos de identidad.

**Conclusión:** la familia del navegador separa grupos con riesgo claramente distinto y refuerza lo visto con el sistema operativo: los dispositivos móviles concentran más fraude que los de escritorio.

In [ ]:
print(df["os_family"].unique())
df[df["split"] == "train"].groupby("os_family", dropna=False)["isFraud"].mean()

[nan 'android' 'ios' 'mac' 'windows' 'linux' 'func' 'other']


os_family
android   0.086
func      0.000
ios       0.061
linux     0.082
mac       0.018
other     0.333
windows   0.035
NaN       0.034
Name: isFraud, dtype: float64

**device_brand**

In [32]:
df["device_brand"] = df["DeviceInfo"].str.split().str[0]
df.loc[df["device_brand"].str.startswith("sm-", na=False), "device_brand"] = "samsung"
df.loc[df["device_brand"].str.startswith(("rv:", "trident"), na=False), "device_brand"] = "desktop_other"

counts = df[df["split"] == "train"]["device_brand"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["device_brand"].isin(keep) & df["device_brand"].notna(), "device_brand"] = "other"

df[df["split"] == "train"].groupby("device_brand", dropna=False)["isFraud"].agg(["mean", "count"])

,mean,count
device_brand,,
desktop_other,0.034,10263
huawei,0.101,605
ios,0.061,17083
macos,0.019,10843
moto,0.163,2103
other,0.139,9459
samsung,0.118,9458
windows,0.062,39722
NaN,0.026,372893


## Marca del dispositivo

`DeviceInfo` identifica el dispositivo con el que se hizo la compra. Aplicamos la misma técnica que con el navegador: primera palabra y agrupación de las marcas con menos de 500 transacciones en `other`.

Antes de agrupar, corregimos dos casos en los que la regla de la primera palabra no funciona:

- **Modelos de Samsung:** cada modelo aparece con su propio código (`sm-j700m`, `sm-g610m`…). Por separado, casi ninguno llega a 500 transacciones, por lo que la mayoría acababan en `other` en lugar de contarse como Samsung. Asignamos a `samsung` todos los valores que empiezan por `sm-`. Con esta corrección, Samsung pasa de 1.900 a 9.458 transacciones.
- **Identificadores de navegador:** `rv:11.0`, `rv:57.0` y `trident/7.0` no son marcas, sino fragmentos técnicos del navegador de un ordenador (Internet Explorer y Firefox). Esa información ya está en `browser_family`, así que los agrupamos en `desktop_other`.

Resultados en entrenamiento:

- **Motorola (16,3%), `other` (13,9%), Samsung (11,8%) y Huawei (10,1%)**, todas marcas de móviles Android o modelos poco comunes, triplican o más la media del 3,5%.
- **iOS (6,1%) y Windows (6,2%)** están por encima de la media.
- **`desktop_other` (3,4%)** está en la media y **macOS (1,9%)** queda por debajo.
- Las transacciones **sin dispositivo (2,6%)** coinciden con las que no tienen datos de identidad.

**Conclusión:** la marca del dispositivo confirma el patrón de las variables anteriores: los móviles, especialmente los Android, concentran más fraude que los ordenadores.

**p_email_provider**

In [33]:
df["p_email_provider"] = df["P_emaildomain"].str.split(".").str[0]

counts = df[df["split"] == "train"]["p_email_provider"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["p_email_provider"].isin(keep) & df["p_email_provider"].notna(), "p_email_provider"] = "other"

df[df["split"] == "train"].groupby("p_email_provider", dropna=False)["isFraud"].agg(["mean", "count"])

,mean,count
p_email_provider,,
anonymous,0.023,29961
aol,0.022,22618
att,0.006,3133
bellsouth,0.028,1580
charter,0.023,688
comcast,0.030,6496
cox,0.021,1121
gmail,0.044,183101
hotmail,0.052,37736


## Proveedor del email del comprador

`P_emaildomain` contiene el dominio del email del comprador. Agrupamos los dominios por proveedor quedándonos con la parte anterior al primer punto (`yahoo.com` y `yahoo.com.mx` pasan a `yahoo`), ya que el país del dominio dispersa las transacciones en grupos demasiado pequeños. Después aplicamos la misma agrupación en `other` de los proveedores con menos de 500 transacciones.

Resultados en entrenamiento:

- **Outlook (9,5%)** casi triplica la media del 3,5%, y **Hotmail (5,2%)** también la supera. Son cuentas gratuitas que se crean y se abandonan con facilidad.
- **Los proveedores de internet de EE. UU.** (`att`, `sbcglobal`, `verizon`, `optonline`, entre el 0,5% y el 1%) tienen entre 4 y 7 veces menos fraude que la media. Estos emails están ligados a un contrato real con el proveedor.
- **Gmail (4,4%)**, que concentra el mayor volumen, y **`anonymous` (2,3%)** están cerca de la media.

**Conclusión:** la señal es más débil que la del dispositivo o el navegador, pero el proveedor permite identificar tanto grupos de riesgo como grupos especialmente seguros, así que lo incluimos como variable.

**Dates**

In [34]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["weekday"] = (df["TransactionDT"] // 86400) % 7

df[df["split"] == "train"].groupby("hour")["isFraud"].agg(["mean", "count"])

,mean,count
hour,,
0,0.032,30557
1,0.032,26772
2,0.037,21922
3,0.036,17102
4,0.050,12603
5,0.067,8392
6,0.074,5065
7,0.101,3156
8,0.097,2181


## Variables de tiempo

`TransactionDT` está en segundos desde un momento de referencia que Vesta no revela, así que no la convertimos a fecha: cualquier fecha de inicio sería una suposición. Calculamos la posición dentro del ciclo con división entera y resto:

- `hour = (TransactionDT // 3600) % 24`
- `weekday = (TransactionDT // 86400) % 7`

No creamos el año ni el mes: los datos cubren unos 6 meses y el test es posterior, por lo que sus valores no aparecerían nunca en entrenamiento. La hora y el día de la semana sí se repiten en ambos conjuntos.

Como no conocemos la hora de inicio, la hora 0 no es necesariamente medianoche. Aun así, la variable es interpretable gracias al volumen de transacciones:

- Las horas **7-9** tienen el **menor volumen** (2.000-3.000 transacciones, frente a más de 33.000 en las horas punta), así que corresponden a la madrugada real.
- En esas mismas horas el fraude alcanza su **máximo (10-10,6%)**, el triple de la media del 3,5%.
- Las horas **13-15** son las de **menor fraude** (2,1-2,5%).

**Conclusión:** el fraude se concentra en las horas de menor actividad, cuando hay menos compras legítimas que lo diluyan. La hora del día es una variable útil para el modelo.

In [36]:
df[df["split"] == "train"].groupby("weekday")["isFraud"].agg(["mean", "count"])

,mean,count
weekday,,
0,0.038,68635
1,0.035,80059
2,0.037,64926
3,0.037,55840
4,0.032,68191
5,0.033,66991
6,0.034,67787


El **día de la semana** no muestra diferencias (entre el 3,2% y el 3,8%, todos cerca de la media). La mantenemos por ahora, ya que podría aportar en combinación con otras variables, como la hora. Decidiremos si se conserva en la fase de modelado, a partir de la importancia que le asigne el modelo.

**decimales**

In [37]:
df["amt_cents"] = ((df["TransactionAmt"] % 1) * 100).round().astype("int8")

(df[df["split"] == "train"]
 .groupby("amt_cents")["isFraud"]
 .agg(["mean", "count"])
 .sort_values("count", ascending=False)
 .head(15))

,mean,count
amt_cents,,
0,0.036,244974
95,0.007,132133
50,0.020,17878
97,0.020,6570
99,0.023,4887
47,0.025,1814
36,0.092,1169
71,0.130,1014
43,0.068,965


## Céntimos del importe

Extraemos la parte decimal del importe en céntimos (`amt_cents`). Para el modelo, `49.99` y `50.00` son casi el mismo valor, pero los céntimos pueden revelar el tipo de compra: un precio de tienda, una cifra redonda o un importe convertido desde otra moneda. Un árbol necesitaría cientos de cortes en `TransactionAmt` para captar esta diferencia; con `amt_cents`, le basta uno.

Resultados en entrenamiento (15 valores más frecuentes):

- **`,95` (0,7%)**, con más de 130.000 transacciones, tiene cinco veces menos fraude que la media del 3,5%. Otros precios de tienda (`,50`, `,97`, `,99`) también quedan por debajo (2,0-2,3%).
- **`,00` (3,6%)**: las cifras redondas están en la media.
- **Céntimos irregulares** (`,11`, `,36`, `,53`, `,71`…) multiplican la media por tres o cuatro (9-14%). Probablemente corresponden a importes convertidos desde otra moneda, es decir, a compras internacionales.

**Conclusión:** los céntimos separan compras de riesgo muy distinto, una información que el importe por sí solo no hace accesible al modelo.